# Project 1 Module 4: Transform Practice

Build confidence with the real helper behavior in `src/transform.py` through prediction, implementation, checking, and debugging.

**Safety boundary:** every GeoDataFrame in this workbook is deterministic and in memory. Do not call `run_transform()` or `process_dataset()`, and do not write to project data paths.

## Working Method

1. Predict before running an answer cell.
2. Replace only the marked `TODO` values or bodies.
3. Run the answer, then its separate check.
4. Treat a failed assertion as evidence about the helper contract.
5. Keep the production module unchanged.

Some answer cells are intentionally unfinished but syntactically valid. `Run All` is therefore not the right workflow until you finish them.

In [ ]:
from pathlib import Path
import re
import sys
import tempfile

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon

project_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "src" / "transform.py").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.transform import (
    TARGET_CRS,
    cast_id_to_string,
    clean_geometry,
    ensure_crs_and_reproject,
    keep_required_fields,
    normalize_col_name,
    normalize_columns,
    processed_output_path,
    repair_geom,
    )


def passed(exercise: str) -> None:
    print(f"PASS: {exercise}")


print("Transform practice environment ready.")

ModuleNotFoundError: No module named 'src'

In [ ]:
def fresh_places() -> gpd.GeoDataFrame:
    """Return a new deterministic fixture so exercises cannot leak mutations."""
    return gpd.GeoDataFrame(
        {
            "Site ID": [101, 102, 103],
            "Display-Name": ["North", "Central", "South"],
            "geometry": [Point(-114.1, 51.05), Point(-114.0, 51.04), Point(-113.9, 51.03)],
        },
        crs="EPSG:4326",
    )


HELPER_ORDER = [
    "normalize_columns",
    "keep_required_fields (only when configured fields exist)",
    "cast_id_to_string",
    "clean_geometry",
    "ensure_crs_and_reproject",
]

assert fresh_places().crs.to_epsg() == 4326
print("Deterministic fixture ready.")

# Part 1: Column Normalization

## Exercise 1: Predict the regex pipeline

For each input, predict the exact result of `normalize_col_name`:

- `"  Road Name  "`
- `"Class---Code"`
- `"A__B / C"`
- `"Étage #2"`

The verified pipeline is: strip and lowercase, replace each run of non-ASCII-alphanumeric characters with `_`, collapse repeated `_`, then trim edge underscores. Notice that `[a-z0-9]` does not retain `é`.

In [ ]:
names_1 = ["  Road Name  ", "Class---Code", "A__B / C", "Étage #2"]

# TODO: replace the empty list with four predicted strings before calling the helper.
prediction_1 = []

actual_1 = [normalize_col_name(name) for name in names_1]
print("prediction:", prediction_1)
print("actual:    ", actual_1)

In [ ]:
expected_1 = ["road_name", "class_code", "a_b_c", "tage_2"]
assert prediction_1 == expected_1, "Write your prediction before relying on actual_1."
assert actual_1 == expected_1
assert normalize_col_name("___Already__Clean___") == "already_clean"
passed("Exercise 1: normalize_col_name regex")

## Exercise 2: Normalize columns without mutating the caller

Predict both column lists after `result_2 = normalize_columns(original_2)`. Then complete the answer. The helper starts with `gdf.copy()`, so object identity and caller columns matter as much as the normalized names.

In [ ]:
original_2 = fresh_places()
predicted_original_columns_2 = []  # TODO: write the three original names.
predicted_result_columns_2 = []  # TODO: write the three normalized names.

result_2 = normalize_columns(original_2)

In [ ]:
assert predicted_original_columns_2 == ["Site ID", "Display-Name", "geometry"]
assert predicted_result_columns_2 == ["site_id", "display_name", "geometry"]
assert list(original_2.columns) == predicted_original_columns_2
assert list(result_2.columns) == predicted_result_columns_2
assert result_2 is not original_2
assert result_2.geometry is not original_2.geometry
passed("Exercise 2: normalize_columns copy behavior")

In [ ]:
# Exercise 3: processed_output_path prediction.
path_inputs_3 = [
    "data/raw/roads.geojson",
    "archive/data/raw/roads.geojson",
    "roads.geojson",
]
# TODO: predict Path values. The helper uses string replacement, not path-part validation.
prediction_3 = []
actual_3 = [processed_output_path(value) for value in path_inputs_3]
print(actual_3)

In [ ]:
expected_3 = [
    Path("data/processed/roads.geojson"),
    Path("archive/data/processed/roads.geojson"),
    Path("roads.geojson"),
]
assert prediction_3 == expected_3
assert actual_3 == expected_3
with tempfile.TemporaryDirectory() as temp_dir:
    isolated_candidate = Path(temp_dir) / actual_3[0].name
    assert isolated_candidate.parent == Path(temp_dir)
passed("Exercise 3: processed_output_path")

# Part 2: Schema and IDs

## Exercise 4: Keep required fields

Start from normalized columns. Predict which requested field is created, what values it contains, and the exact output column order. Remember: `geometry` is appended by the helper, so callers must remove it from `keep_fields` first.

In [ ]:
source_4 = normalize_columns(fresh_places())
required_4 = ["site_id", "display_name", "sector"]

# TODO: call keep_required_fields and unpack both return values.
kept_4 = source_4
missing_4 = []

In [ ]:
assert missing_4 == ["sector"], "Missing fields preserve requested-field order."
assert list(kept_4.columns) == ["site_id", "display_name", "sector", "geometry"]
assert kept_4["sector"].isna().all()
assert "sector" not in source_4.columns, "The helper should not mutate its input."
assert kept_4 is not source_4
passed("Exercise 4: keep_required_fields")

## Exercise 5: ID casting and short circuits

Predict the dtype and values after casting `site_id`. Then test both no-op branches: `id_field=None` and a field name that is absent. All three calls return copies, even when no column is changed. Pandas nullable string conversion preserves missing values as `<NA>`.

In [ ]:
ids_5 = gpd.GeoDataFrame(
    {"site_id": [7, None], "geometry": [Point(0, 0), Point(1, 1)]},
    crs="EPSG:4326",
)

# TODO: call the helper for a real ID, None, and an absent field.
cast_5 = ids_5
none_5 = ids_5
absent_5 = ids_5

In [ ]:
assert str(cast_5["site_id"].dtype).startswith("string")
assert cast_5["site_id"].iloc[0] == "7.0"
assert pd.isna(cast_5["site_id"].iloc[1])
assert none_5.equals(ids_5) and none_5 is not ids_5
assert absent_5.equals(ids_5) and absent_5 is not ids_5
assert ids_5["site_id"].dtype != cast_5["site_id"].dtype
passed("Exercise 5: ID casting and short circuits")

# Part 3: Geometry

## Exercise 6: Boolean masks for null, empty, and invalid geometry

Create three masks independently before cleaning. Use `geometry.notnull()`, `geometry.is_empty`, and GeoDataFrame `is_valid`. A null geometry is also reported invalid by GeoPandas, which is why `clean_geometry` removes null and empty rows before counting `invalid_before`.

In [ ]:
bowtie_6 = Polygon([(0, 0), (2, 2), (2, 0), (0, 2), (0, 0)])
geometry_6 = gpd.GeoDataFrame(
    {"kind": ["valid", "null", "empty", "invalid"]},
    geometry=[Point(0, 0), None, Point(), bowtie_6],
    crs="EPSG:4326",
)

# TODO: replace these placeholder Series with the three requested masks.
non_null_6 = pd.Series(False, index=geometry_6.index)
empty_6 = pd.Series(False, index=geometry_6.index)
invalid_6 = pd.Series(False, index=geometry_6.index)

In [ ]:
assert non_null_6.tolist() == [True, False, True, True]
assert empty_6.tolist() == [False, False, True, False]
assert invalid_6.tolist() == [False, True, False, True]
survives_precheck_6 = non_null_6 & ~empty_6
assert geometry_6.loc[survives_precheck_6, "kind"].tolist() == ["valid", "invalid"]
passed("Exercise 6: geometry masks")

## Exercise 7: Repair and clean geometry

Predict whether the bow-tie polygon is valid before and after `repair_geom`. Then pass the four-row fixture through `clean_geometry` and predict `invalid_before`, `invalid_after`, surviving labels, and whether the input was mutated.

### Debugging clue

If your count is `2` instead of `1`, you counted invalidity before removing null/empty geometries. If empty rows survive, check the `~` inversion on `is_empty`.

In [ ]:
repaired_7 = repair_geom(bowtie_6)

# TODO: call clean_geometry and unpack all three return values.
cleaned_7 = geometry_6
invalid_before_7 = -1
invalid_after_7 = -1

print("bow-tie valid before/after:", bowtie_6.is_valid, repaired_7.is_valid)

In [ ]:
assert bowtie_6.is_valid is False
assert repaired_7 is not None and repaired_7.is_valid
assert invalid_before_7 == 1
assert invalid_after_7 == 0
assert cleaned_7["kind"].tolist() == ["valid", "invalid"]
assert len(geometry_6) == 4, "clean_geometry should work on a copy."
assert cleaned_7.geometry.notnull().all()
assert (~cleaned_7.geometry.is_empty).all()
assert cleaned_7.is_valid.all()
passed("Exercise 7: repair_geom and clean_geometry")

# Part 4: CRS Semantics

## Exercise 8: `set_crs` versus `to_crs`

`set_crs` assigns coordinate meaning without changing coordinate numbers. `to_crs` transforms coordinate numbers into a new reference system. The production helper sets missing CRS to `EPSG:4326`, then always calls `to_crs(target_crs)`.

Predict the coordinates after each operation before completing the code.

In [ ]:
unknown_8 = gpd.GeoDataFrame(
    {"name": ["Calgary"]},
    geometry=[Point(-114.0719, 51.0447)],
)

# TODO: assign EPSG:4326 metadata, then transform a separate result to TARGET_CRS.
assigned_8 = unknown_8
projected_8 = unknown_8

In [ ]:
source_xy_8 = (-114.0719, 51.0447)
assert unknown_8.crs is None
assert assigned_8.crs.to_epsg() == 4326
assert (assigned_8.geometry.x.iloc[0], assigned_8.geometry.y.iloc[0]) == source_xy_8
assert projected_8.crs.to_string() == TARGET_CRS
assert projected_8.geometry.x.iloc[0] != source_xy_8[0]
assert projected_8.geometry.y.iloc[0] != source_xy_8[1]

via_helper_8 = ensure_crs_and_reproject(unknown_8, TARGET_CRS)
assert via_helper_8.crs == projected_8.crs
assert via_helper_8.geometry.iloc[0].equals_exact(projected_8.geometry.iloc[0], 0.001)
passed("Exercise 8: set_crs versus to_crs")

# Part 5: Orchestration Evidence

## Exercise 9: Build a transform log row and order the process

Construct the same 12-field row shape initialized by `process_dataset`, but use a fixed timestamp for deterministic practice. Then arrange the helper calls in their verified order.

The full production flow checks file existence, reads data, records `rows_in`, applies the helper pipeline, creates the output directory, writes GeoJSON, and records `rows_out`. This exercise stops before all filesystem actions.

In [ ]:
# TODO: fill every value using the production row defaults and these deterministic inputs.
log_row_9 = {
    "dataset": "practice_sites",
}

# TODO: reorder these strings to match process_dataset.
process_order_9 = [
    "ensure_crs_and_reproject",
    "clean_geometry",
    "cast_id_to_string",
    "keep_required_fields",
    "normalize_columns",
]

In [ ]:
expected_keys_9 = [
    "dataset", "input_path", "output_path", "rows_in", "rows_out",
    "missing_fields", "invalid_before", "invalid_after", "crs_out",
    "processed_at_utc", "status", "error",
]
assert list(log_row_9) == expected_keys_9, "Preserve DictWriter field order."
assert log_row_9 == {
    "dataset": "practice_sites",
    "input_path": "data/raw/practice.geojson",
    "output_path": "data/processed/practice.geojson",
    "rows_in": 0,
    "rows_out": 0,
    "missing_fields": "",
    "invalid_before": 0,
    "invalid_after": 0,
    "crs_out": TARGET_CRS,
    "processed_at_utc": "2030-01-02T03:04:05+00:00",
    "status": "ok",
    "error": "",
}
assert process_order_9 == HELPER_ORDER
passed("Exercise 9: log row and process ordering")

In [ ]:
# Debugging drills: each function is syntactically valid but behaviorally wrong.
# Fix one bug at a time, then write a tiny assertion that would have caught it.

def buggy_normalize_10(name: str) -> str:
    name = name.strip()
    return re.sub(r"[^a-z0-9]+", "_", name)  # TODO: find two missing steps.


def buggy_keep_10(gdf: gpd.GeoDataFrame, fields: list[str]) -> gpd.GeoDataFrame:
    for field in fields:
        if field not in gdf.columns:
            gdf[field] = None  # TODO: protect the caller.
    return gdf[fields]  # TODO: retain spatial behavior.


def buggy_crs_10(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    return gdf.set_crs(TARGET_CRS)  # TODO: explain why this can mislabel Calgary lon/lat.


debug_notes_10 = {
    "buggy_normalize_10": "TODO",
    "buggy_keep_10": "TODO",
    "buggy_crs_10": "TODO",
}

# Capstone: Mini In-Memory Transform

Complete `mini_transform` by composing the verified helpers. Requirements:

- normalize columns first;
- optionally retain requested fields and create missing fields;
- cast the ID only when configured and present;
- clean geometry and collect both invalid counts;
- assign missing source CRS as `EPSG:4326` and reproject to the target;
- return `(transformed_gdf, report_dict)`;
- never call `to_file`, `process_dataset`, `run_transform`, or a logging function.

In [ ]:
def mini_transform(
    gdf: gpd.GeoDataFrame,
    keep_fields: list[str],
    id_field: str | None,
    target_crs: str = TARGET_CRS,
) -> tuple[gpd.GeoDataFrame, dict]:
    """Transform one in-memory layer and return compact QA evidence."""
    # TODO: replace this placeholder body by composing the imported helpers.
    report = {
        "rows_in": len(gdf),
        "rows_out": len(gdf),
        "missing_fields": [],
        "invalid_before": 0,
        "invalid_after": 0,
        "crs_out": str(gdf.crs) if gdf.crs is not None else None,
    }
    return gdf.copy(), report

In [ ]:
capstone_input = gpd.GeoDataFrame(
    {
        "Site ID": [1, 2, 3, 4],
        "Label": ["valid", "null", "empty", "repair"],
    },
    geometry=[Point(-114.1, 51.05), None, Point(), bowtie_6],
)
capstone_output, capstone_report = mini_transform(
    capstone_input,
    keep_fields=["site_id", "label", "sector"],
    id_field="site_id",
)

assert list(capstone_input.columns) == ["Site ID", "Label", "geometry"]
assert list(capstone_output.columns) == ["site_id", "label", "sector", "geometry"]
assert capstone_output["label"].tolist() == ["valid", "repair"]
assert capstone_output["site_id"].tolist() == ["1", "4"]
assert capstone_output["sector"].isna().all()
assert capstone_output.crs.to_string() == TARGET_CRS
assert capstone_output.geometry.notnull().all()
assert (~capstone_output.geometry.is_empty).all()
assert capstone_output.is_valid.all()
assert capstone_report == {
    "rows_in": 4,
    "rows_out": 2,
    "missing_fields": ["sector"],
    "invalid_before": 1,
    "invalid_after": 0,
    "crs_out": TARGET_CRS,
}
passed("Capstone: mini_transform")

# Bridge Back, Optional References, and Review

## Bridge-back dictionary

Use this as a retrieval map from workbook ideas to production responsibilities:

```python
BRIDGE_BACK = {
    "predictable schema": "normalize_col_name / normalize_columns",
    "required output schema": "keep_required_fields",
    "join-safe identifiers": "cast_id_to_string",
    "geometry QA": "clean_geometry / repair_geom",
    "shared spatial reference": "ensure_crs_and_reproject",
    "dataset evidence": "process_dataset log row",
    "batch orchestration": "run_transform",
}
```

<details>
<summary>Optional reference solutions (open only after a genuine attempt)</summary>

```python
# Exercise 1
prediction_1 = ["road_name", "class_code", "a_b_c", "tage_2"]

# Exercise 2
predicted_original_columns_2 = ["Site ID", "Display-Name", "geometry"]
predicted_result_columns_2 = ["site_id", "display_name", "geometry"]

# Exercise 3
prediction_3 = [Path("data/processed/roads.geojson"),
                Path("archive/data/processed/roads.geojson"), Path("roads.geojson")]

# Exercise 4
kept_4, missing_4 = keep_required_fields(source_4, required_4)

# Exercise 5
cast_5 = cast_id_to_string(ids_5, "site_id")
none_5 = cast_id_to_string(ids_5, None)
absent_5 = cast_id_to_string(ids_5, "missing_id")

# Exercise 6
non_null_6 = geometry_6.geometry.notnull()
empty_6 = geometry_6.geometry.is_empty
invalid_6 = ~geometry_6.is_valid

# Exercise 7
cleaned_7, invalid_before_7, invalid_after_7 = clean_geometry(geometry_6)

# Exercise 8
assigned_8 = unknown_8.set_crs("EPSG:4326")
projected_8 = assigned_8.to_crs(TARGET_CRS)

# Exercise 9
process_order_9 = HELPER_ORDER.copy()

# Capstone core
result = normalize_columns(gdf)
result, missing = keep_required_fields(result, keep_fields) if keep_fields else (result, [])
result = cast_id_to_string(result, id_field)
result, before, after = clean_geometry(result)
result = ensure_crs_and_reproject(result, target_crs)
```
</details>

## Review schedule

- **Today:** finish Exercises 1-5 and explain each failed check.
- **Tomorrow:** redo geometry and CRS exercises from a fresh kernel.
- **In 3 days:** repair all debugging drills without opening references.
- **In 7 days:** write the capstone from a blank function body.
- **In 14 days:** trace `process_dataset` aloud, naming each state change and log update.

Mastery check: you can explain why helper order matters, predict copy/no-op behavior, and distinguish assigning a CRS from transforming coordinates without running code first.

## AI Practice Review

After completing the working copy, save it and ask Copilot:

> Review my completed practice notebook without assigning a progression grade. Preserve my original answers and code. Inspect my reasoning, implementations, self-checks, errors, and outputs. Cite evidence for demonstrated strengths and misconceptions, recommend the smallest useful exercises to retry, and ask targeted follow-up questions before giving complete corrected answers.

After reviewing the feedback, preserve the completed notebook with `python scripts/save_attempt.py 4`.